# 08 · Cell-cell communication with LIANA+ — GSE216146

Ligand-receptor inference between cell types, chemotherapy (cisplatin) vs control, with **LIANA+** (`liana` 2.1.1,
mouse resource `mouseconsensus`). Three complementary views:

1. **Consensus per sample:** `rank_aggregate` over six methods (CellPhoneDB, Connectome, log2FC, NATMI,
   SingleCellSignalR and **CellChat**) run separately for every biosample. An interaction is "significant" in a sample
   if its consensus `specificity_rank` <= `communication.specificity_rank`. The number of significant interactions per
   sender -> receiver pair is compared between conditions per sample (OLS `~ batch + condition`).
2. **CellChat method per condition** (CellChat-style comparison: number and total probability of significant
   interactions per sender -> receiver pair). Cells are pooled per condition, so this is descriptive.
3. **Differential interactions from the pseudobulk DE (05):** `li.mt.df_to_lr` joins the DESeq2 statistics of the
   ligand (in the sender) and the receptor (in the receiver). Interactions where both partners change in the same
   direction are the best-supported changes, because the replicate is the sample. Genes flagged in 05 are removed.

Ribosomal and mitochondrial genes are removed from the resource (`exclude_gene_regex`). Only the cell types tested in 05 are used as senders/receivers in view 3; low-quality microglia are excluded everywhere.

**Input:** `results/03_annotated.h5ad`, `results/de/*.tsv` · **Output:** `results/communication/*.tsv`,
tables `results/tables/08_*`, plots `results/plots/08_*`.

In [ ]:
import gc
import re
import sys
import warnings
from importlib.metadata import version
from pathlib import Path

import liana as li
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import sc_utils as scu

cfg = scu.load_config(ROOT / "config.yaml")
p = cfg["communication"]
out_dir = cfg["paths"]["results_dir"] / "communication"
out_dir.mkdir(exist_ok=True)
scu.setup_plotting()
pd.set_option("display.max_rows", 200)
# LIANA prints informative warnings for every run (empty features, excluded small groups, AnnData views)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", module="anndata")
print("liana", version("liana"))
p

In [ ]:
adata = sc.read_h5ad(cfg["paths"]["results_dir"] / "03_annotated.h5ad")
adata = adata[~adata.obs["cell_type"].isin(cfg["downstream"]["exclude_cell_types"])].copy()
adata.obs["cell_type"] = adata.obs["cell_type"].cat.remove_unused_categories()
del adata.layers["counts"], adata.obsp
gc.collect()

meta = (adata.obs[["biosample", "condition", "batch"]].drop_duplicates().set_index("biosample")
        .sort_values(["condition", "batch"]))
meta.index = meta.index.astype(str)
meta["condition"] = meta["condition"].cat.remove_unused_categories()
meta["batch"] = meta["batch"].astype(str)
ref, alt = list(meta["condition"].cat.categories)
types = list(adata.obs["cell_type"].cat.categories)
common = dict(groupby="cell_type", resource_name=p["resource"], expr_prop=p["expr_prop"], min_cells=p["min_cells"],
              use_raw=False, n_perms=p["n_perms"], seed=p["seed"], n_jobs=p["n_jobs"], verbose=False)
adata

## 1. Consensus (6 methods incl. CellChat) per sample

In [ ]:
consensus = li.mt.AggregateClass(li.mt.aggregate_meta, methods=[getattr(li.mt, m) for m in p["methods"]])
print([m.method_name for m in consensus.methods])
# Ribosomal / mt genes are removed from the resource (Rpsa "laminin receptor", Rps19 ligand: technical, not signalling)
drop_rx = re.compile(p["exclude_gene_regex"])
resource = li.rs.select_resource(p["resource"])
keep = ~resource["ligand"].str.split("_").map(lambda g: any(drop_rx.search(x) for x in g)) & \
       ~resource["receptor"].str.split("_").map(lambda g: any(drop_rx.search(x) for x in g))
resource = resource[keep]
common = {k: v for k, v in common.items() if k != "resource_name"} | {"resource": resource}
print("resource:", int(keep.sum()), "of", len(keep), "pairs kept")
consensus.by_sample(adata, sample_key="biosample", key_added="liana_by_sample", **common)
res_s = adata.uns.pop("liana_by_sample")
res_s["biosample"] = res_s["biosample"].astype(str)
res_s.to_csv(out_dir / "consensus_by_sample.tsv.gz", sep="\t", index=False)
res_s.shape

In [ ]:
sig = res_s[res_s["specificity_rank"] <= p["specificity_rank"]]
n_sig = sig.groupby(["biosample", "source", "target"], observed=True).size().rename("n").reset_index()
n_sig["pair"] = n_sig["source"].astype(str) + " -> " + n_sig["target"].astype(str)
wide = n_sig.pivot(index="biosample", columns="pair", values="n").reindex(meta.index).fillna(0)
# A pair can only be compared where both cell types were present in a sample (>= min_cells); others -> NaN
present = adata.obs.groupby(["biosample", "cell_type"], observed=True).size().unstack(fill_value=0) >= p["min_cells"]
present.index = present.index.astype(str)
for pair in wide.columns:
    s, t = pair.split(" -> ")
    wide.loc[~(present[s] & present[t]).reindex(wide.index).to_numpy(), pair] = np.nan

testable = wide.columns[wide.notna().sum() >= p["min_samples_test"]]
pair_test = scu.ols_by_column(wide[testable], meta, p["formula"]).sort_values("pval")
pair_test.index.name = "pair"
pair_test.to_csv(cfg["paths"]["tables_dir"] / "08_consensus_pair_counts_test.tsv", sep="\t")
print("total significant interactions per sample:")
print(sig.groupby("biosample").size().reindex(meta.index).to_frame("n").join(meta))
pair_test.head(15).round(3)

In [ ]:
# Mean number of significant interactions per sender -> receiver: control, chemotherapy, difference
mean_cond = {c: wide.loc[meta.index[meta["condition"] == c]].mean() for c in (ref, alt)}
def to_matrix(s):
    m = s.rename("v").reset_index()
    m[["source", "target"]] = m["pair"].str.split(" -> ", expand=True)
    return m.pivot(index="source", columns="target", values="v").reindex(index=types, columns=types)

fig, axes = plt.subplots(1, 3, figsize=(21, 6.5))
vmax = max(to_matrix(mean_cond[c]).max().max() for c in (ref, alt))
for ax, c in zip(axes[:2], (ref, alt)):
    sns.heatmap(to_matrix(mean_cond[c]), ax=ax, cmap="viridis", vmin=0, vmax=vmax, cbar_kws={"label": "interactions"})
    ax.set_title(f"{c}: significant interactions (mean per sample)")
# difference only for pairs present in all samples (others are noise from 2 vs 2 samples)
diff = to_matrix((mean_cond[alt] - mean_cond[ref])[testable])
lim = np.nanmax(np.abs(diff.to_numpy()))
star = to_matrix(pair_test["pval"]).map(lambda v: "*" if v < 0.05 else "")
sns.heatmap(diff, ax=axes[2], cmap="RdBu_r", center=0, vmin=-lim, vmax=lim, annot=star, fmt="",
            cbar_kws={"label": f"{alt} - {ref}"})
axes[2].set_title("Difference, pairs present in all 6 samples (* nominal p < 0.05)")
for ax in axes:
    ax.set_xlabel("receiver (target)"); ax.set_ylabel("sender (source)")
fig.tight_layout()
scu.save_fig(fig, cfg, "08_consensus_pair_counts")
plt.show()

## 2. CellChat method per condition (descriptive)

In [ ]:
cc = []
for cond in (ref, alt):
    sub = adata[adata.obs["condition"] == cond].copy()
    li.mt.cellchat(sub, key_added="cc", **common)
    cc.append(sub.uns["cc"].assign(condition=cond))
    del sub
gc.collect()
cc = pd.concat(cc, ignore_index=True)
cc.to_csv(out_dir / "cellchat_by_condition.tsv.gz", sep="\t", index=False)
cc_sig = cc[cc["cellchat_pvals"] < p["cellchat_pval"]]
cc_sum = cc_sig.groupby(["condition", "source", "target"], observed=True).agg(n=("lr_probs", "size"), strength=("lr_probs", "sum")).reset_index()
cc_sum.groupby("condition")[["n", "strength"]].sum()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
for ax, val in zip(axes, ["n", "strength"]):
    m = {c: cc_sum[cc_sum["condition"] == c].pivot(index="source", columns="target", values=val)
               .reindex(index=types, columns=types).fillna(0) for c in (ref, alt)}
    d = m[alt] - m[ref]
    lim = np.abs(d.to_numpy()).max()
    sns.heatmap(d, ax=ax, cmap="RdBu_r", center=0, vmin=-lim, vmax=lim, cbar_kws={"label": f"{alt} - {ref}"})
    ax.set_title(f"CellChat: differential {'number' if val == 'n' else 'strength (sum lr_probs)'} of interactions")
    ax.set_xlabel("receiver (target)"); ax.set_ylabel("sender (source)")
fig.tight_layout()
scu.save_fig(fig, cfg, "08_cellchat_differential")
plt.show()

## 3. Differential interactions from the pseudobulk DE (05)

`interaction_stat` = mean of the ligand Wald statistic (in the sender) and the receptor Wald statistic (in the
receiver). "Concordant" = both partners change in the same direction, one with padj < `de_padj` and the other
with nominal p < `de_pval`.

In [ ]:
summary = pd.read_csv(cfg["paths"]["tables_dir"] / "05_de_summary.tsv", sep="\t", index_col=0)
tested = list(summary.index)
dea = []
for ct in tested:
    r = pd.read_csv(cfg["paths"]["results_dir"] / "de" / f"{scu.slug(ct)}.tsv", sep="\t", index_col=0)
    r = r[r["flag"].isna()][["stat", "log2FoldChange", "pvalue", "padj"]].assign(cell_type=ct)
    dea.append(r)
dea = pd.concat(dea)
dea[["pvalue", "padj"]] = dea[["pvalue", "padj"]].fillna(1.0)

sub = adata[adata.obs["cell_type"].isin(tested)].copy()
sub.obs["cell_type"] = sub.obs["cell_type"].cat.remove_unused_categories()
sub = sub[:, sub.var_names.isin(dea.index.unique())].copy()
lr = li.mt.df_to_lr(sub, dea_df=dea, groupby="cell_type", stat_keys=["stat", "log2FoldChange", "pvalue", "padj"],
                    resource=resource, expr_prop=p["expr_prop"], min_cells=p["min_cells"],
                    use_raw=False, verbose=False)
del sub
lr["interaction_stat"] = lr[["ligand_stat", "receptor_stat"]].mean(axis=1)
lr["concordant"] = ((np.sign(lr["ligand_stat"]) == np.sign(lr["receptor_stat"]))
                    & (lr[["ligand_padj", "receptor_padj"]].min(axis=1) < p["de_padj"])
                    & (lr[["ligand_pvalue", "receptor_pvalue"]].max(axis=1) < p["de_pval"]))
lr = lr.sort_values("interaction_stat", key=np.abs, ascending=False)
lr.to_csv(out_dir / "de_interactions.tsv.gz", sep="\t", index=False)
print(lr.shape, "interactions;", int(lr["concordant"].sum()), "concordant")
cols = ["source", "target", "ligand_complex", "receptor_complex", "ligand_stat", "ligand_padj",
        "receptor_stat", "receptor_padj", "interaction_stat"]
lr.loc[lr["concordant"], cols].head(p["n_top"]).round(3)

In [ ]:
# Top concordant interactions: ligand and receptor log2FC
top = lr[lr["concordant"]].head(p["n_top"]).copy()
top["label"] = (top["ligand_complex"] + " -> " + top["receptor_complex"] + "   (" + top["source"].astype(str)
                + " -> " + top["target"].astype(str) + ")")
if not top.empty:
    fig, ax = plt.subplots(figsize=(8, 0.32 * len(top) + 1.5))
    y = np.arange(len(top))[::-1]
    ax.scatter(top["ligand_log2FoldChange"], y, marker="o", s=45, c="#c0392b", label="ligand (sender)")
    ax.scatter(top["receptor_log2FoldChange"], y, marker="s", s=45, c="#2c7bb6", label="receptor (receiver)")
    for yi, (a, b) in zip(y, zip(top["ligand_log2FoldChange"], top["receptor_log2FoldChange"])):
        ax.plot([a, b], [yi, yi], c="0.7", lw=0.8, zorder=0)
    ax.axvline(0, c="0.3", lw=0.8)
    ax.set_yticks(y, top["label"], fontsize=8)
    ax.set_xlabel(f"log2 fold change ({alt} / {ref})")
    ax.set_title("Top concordant ligand-receptor changes (pseudobulk DE)")
    ax.legend(fontsize=8, frameon=False, loc="lower right"); ax.grid(False)
    scu.save_fig(fig, cfg, "08_de_interactions_top")
    plt.show()

In [ ]:
# SASP ligands (SenMayo) among the changed interactions
senmayo = set(next(iter(scu.read_gmt(ROOT / cfg["senescence"]["senmayo_gmt"]).values())))
sasp = lr[lr["ligand_complex"].isin(senmayo) & (lr["ligand_padj"] < p["de_padj"])]
sasp.to_csv(cfg["paths"]["tables_dir"] / "08_sasp_ligand_interactions.tsv", sep="\t", index=False)
(sasp.groupby(["source", "ligand_complex"], observed=True)
     .agg(ligand_log2FC=("ligand_log2FoldChange", "first"), ligand_padj=("ligand_padj", "first"),
          n_receivers=("target", "nunique"), receptors=("receptor_complex", lambda s: ", ".join(sorted(set(s)))))
     .sort_values("ligand_padj").round(4))

In [ ]:
del adata, res_s, cc, lr
_ = gc.collect()